# Digital Clone — Demo

End-to-end demo of the multi-agent system: ask a question as if you emailed
**Kay Mann**, and the system drafts a grounded reply in her voice, grades it, and
either sends it or offers a call.

```
question → draft (style) → evaluate (grounding + style + confidence) → decide (send / reflect / fallback)
```

**Select the `digital-clone` kernel before running.** These cells make live LLM
calls via OpenRouter (needs `OPENROUTER_API_KEY` in `.env`).

## Setup — build the agents

Loads Kay Mann's style profile + exemplars, the cognitive-science FAISS index, and
the OpenRouter client, then wires them into the orchestrator.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "agents")))

from evaluator import Evaluator
from knowledge_agent import KnowledgeAgent
from orchestrator import Orchestrator
from style_agent import StyleAgent, make_client

sa = StyleAgent(name="Kay Mann")  # profile + exemplar emails
kb = KnowledgeAgent().build()  # cognitive-science FAISS index
client = make_client()  # OpenRouter (Claude)
orch = Orchestrator(sa, kb, Evaluator(sa, client), client)
print("ready")

## In-domain question → SEND

A cognitive-science question the knowledge base can answer. The system drafts,
grounds, scores, and sends.

In [ ]:
result = orch.run("How do Bayesian models explain human cognition?")
print("decision  :", result["decision"])
print("confidence:", result["verdict"]["confidence"])
print("attempts  :", result["attempts"])
print("\n" + result["reply"])

## Out-of-domain question → FALLBACK

Nothing in the cognitive-science corpus supports this, so the system declines to
guess and offers a call instead — in Kay's voice.

In [ ]:
result = orch.run("What were Enron's Q3 2001 earnings?")
print("decision  :", result["decision"])
print("confidence:", result["verdict"]["confidence"])
print("\n" + result["reply"])

## Inspect the parts

The style profile, the retrieved evidence, and the style-typicality score are all
available on their own:

In [ ]:
print("Kay's style directives:\n")
print(sa.directives())

print("\nTop retrieved chunk for a query:")
hit = kb.retrieve("what is a connectionist model?", k=1)[0]
print(f"  [{hit['score']:.3f}] {hit['citation']}")

print("\nStyle score of a real Kay email vs off-style text:")
print("  real  :", sa.style_score(sa.exemplars(1)[0]))
print("  physics:", sa.style_score("Quantum chromodynamics exhibits asymptotic freedom."))

## The evaluation harness

Run a labelled question set and report routing accuracy, confidence, style,
grounding, and fallback rate:

```bash
python evaluate.py
```

Or inline (spends tokens — a handful of questions through the full pipeline):

In [ ]:
# from evaluate import main; main()